
#  Practical RAG with **LlamaIndex** (and how it complements LangChain)

**Goal:** replicate the core steps from the LangChain lab using **LlamaIndex**'s data→index→query workflow, while calling out key differences and interop patterns.



## 0) Setup

> This notebook uses a tiny in-notebook corpus so it runs offline.  
> If you haven't installed the libraries, uncomment the `pip install` cells.


In [1]:

# %%capture
# !pip install llama-index-core llama-index-readers-file docx2txt fastapi uvicorn --quiet
# Optional (only if you want to run LangChain comparison cells):
# !pip install langchain langchain-community --quiet
# !pip install -U llama-index-embeddings-huggingface sentence-transformers
# !pip install -U llama-index-llms-huggingface transformers accelerate
# install Ollama
# brew install ollama
# ollama serve    # in a terminal; keeps running

# pull a tiny instruct model (pick one)
# ollama pull qwen2.5:0.5b     # ~0.5B params
# or: ollama pull phi3:mini   # ~3.8B (still pretty light)
# or: ollama pull llama3.2:1b-instruct




from typing import List, Dict
import re, time, json



## 1) Create a tiny corpus (so we can run without external files)

We'll simulate 3 short documents with headers/footers and some duplication so we can demonstrate **normalization**.


In [2]:

toy_docs = {
    "install_guide.md": """
    ACME INTERNAL — DO NOT DISTRIBUTE
    # ACME X — Installation Guide

    ## Requirements
    Memory: 16GB RAM (32GB recommended)
    Disk: 20GB free space
    Notes: Admin access required

    ## Steps
    1) Download ACME X installer
    2) Run setup with admin rights
    3) Reboot after installation

    © 2025 ACME Corp
    """,
    "faq.txt": """
    ACME INTERNAL — DO NOT DISTRIBUTE
    # FAQ — ACME X

    Q: How do I reset my password?
    A: Go to Settings → Security → Reset.

    Q: Where are logs stored?
    A: /var/log/acme-x/app.log

    © 2025 ACME Corp
    """,
    "troubleshooting.md": """
    ACME INTERNAL — DO NOT DISTRIBUTE
    # Troubleshooting ACME X

    If installation fails, ensure admin privileges and 20GB free disk.
    For password reset issues, verify SSO is available.

    © 2025 ACME Corp
    """
}
list(toy_docs.keys())


['install_guide.md', 'faq.txt', 'troubleshooting.md']


## 2) Normalize the text (strip headers/footers, dedupe lines)

**Why:** Clean context improves retrieval quality.  
**What we do:** remove common headers/footers, collapse duplicate lines, keep a stable `doc_id` and later mint `chunk_id`.


In [3]:

import re
HEADER_RE = re.compile(r"^ACME INTERNAL — DO NOT DISTRIBUTE\s*$", re.I)
FOOTER_RE = re.compile(r"^©\s*\d{4}\s*ACME Corp\s*$", re.I)

def normalize(name: str, text: str):
    lines, seen = [], set()
    for raw in text.splitlines():
        line = raw.strip()
        if not line or HEADER_RE.match(line) or FOOTER_RE.match(line):
            continue
        if line in seen:
            continue
        seen.add(line)
        lines.append(line)
    return {
        "doc_id": name,
        "title": name.split(".")[0].replace("_"," ").title(),
        "text": "\n".join(lines)
    }

norm_docs = [normalize(name, txt) for name, txt in toy_docs.items()]
norm_docs[0]


{'doc_id': 'install_guide.md',
 'title': 'Install Guide',
 'text': '# ACME X — Installation Guide\n## Requirements\nMemory: 16GB RAM (32GB recommended)\nDisk: 20GB free space\nNotes: Admin access required\n## Steps\n1) Download ACME X installer\n2) Run setup with admin rights\n3) Reboot after installation'}


## 3) Split into nodes and mint **chunk_id**

**LangChain equivalent:** `RecursiveCharacterTextSplitter` + `metadata` per chunk.  
**LlamaIndex:** use `SentenceSplitter` and attach metadata including stable `chunk_id`.


In [4]:

from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.schema import TextNode

splitter = SentenceSplitter(chunk_size=600, chunk_overlap=80)

def to_nodes(doc):
    nodes = []
    parts = splitter.split_text(doc["text"])
    for i, part in enumerate(parts):
        chunk_id = f"{doc['doc_id']}#chunk_{i:03d}"
        nodes.append(TextNode(text=part, metadata={
            "doc_id": doc["doc_id"], "title": doc["title"], "chunk_id": chunk_id
        }))
    return nodes

all_nodes = []
for d in norm_docs:
    all_nodes.extend(to_nodes(d))

len(all_nodes), all_nodes[0].metadata


(3,
 {'doc_id': 'install_guide.md',
  'title': 'Install Guide',
  'chunk_id': 'install_guide.md#chunk_000'})


## 4) Index, persist, and reload

Index once, **persist**, then **reload** in any process (dev, CI, prod).


In [5]:
# Configure a local embedding model so we don’t need OPENAI_API_KEY
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings

# Small, fast, and good enough for demos
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")


In [6]:

from llama_index.core import VectorStoreIndex, StorageContext, load_index_from_storage

index = VectorStoreIndex(all_nodes)
index.storage_context.persist(persist_dir="index_demo")

storage = StorageContext.from_defaults(persist_dir="index_demo")
reloaded_index = load_index_from_storage(storage)
reloaded_index is not None


Loading llama_index.core.storage.kvstore.simple_kvstore from index_demo/docstore.json.
Loading llama_index.core.storage.kvstore.simple_kvstore from index_demo/index_store.json.


True


## 5) Query: Quick vs Custom paths

- **Quick**: `as_query_engine(top_k, response_mode)`  
- **Custom**: build a `VectorIndexRetriever` (set `k`, filters), add `SimilarityPostprocessor` (cutoff), then `RetrieverQueryEngine`.


In [7]:
# pip install -U llama-index-llms-ollama
from llama_index.llms.ollama import Ollama
from llama_index.core import Settings

# Smallest + fastest choice:
Settings.llm = Ollama(model="qwen2.5:0.5b", request_timeout=60.0, temperature=0.1)
# If you pulled a different model, swap the name accordingly.



In [8]:

from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.postprocessor import SimilarityPostprocessor

qe_quick = reloaded_index.as_query_engine(similarity_top_k=4, response_mode="compact")
resp_quick = qe_quick.query("What are the hardware requirements for ACME X?")
print("QUICK:", str(resp_quick))
print("Sources:", [sn.node.metadata.get("chunk_id") for sn in resp_quick.source_nodes])

retriever = VectorIndexRetriever(index=reloaded_index, similarity_top_k=6)
post = [SimilarityPostprocessor(similarity_cutoff=0.55)]  # calibrate on your corpus
qe_custom = RetrieverQueryEngine.from_args(retriever=retriever, node_postprocessors=post, response_mode="compact")
resp_custom = qe_custom.query("What are the hardware requirements for ACME X?")
print("\nCUSTOM:", str(resp_custom))
print("Sources:", [sn.node.metadata.get("chunk_id") for sn in resp_custom.source_nodes])


QUICK: Memory: 16GB RAM (32GB recommended)
Sources: ['install_guide.md#chunk_000', 'troubleshooting.md#chunk_000', 'faq.txt#chunk_000']

CUSTOM: Memory: 16GB RAM (32GB recommended)
Sources: ['install_guide.md#chunk_000', 'troubleshooting.md#chunk_000', 'faq.txt#chunk_000']



## 6) Grounded prompts & JSON output (plus “I don’t know”)

We’ll nudge the synthesizer to output **JSON** with `answer` and `sources[]`.  
For strictness, parse and validate the output against a schema in production.


In [9]:

from llama_index.core import PromptTemplate

prompt = PromptTemplate(
    "Use ONLY the context to answer. If insufficient, reply: "
    "{\"answer\":\"I don't know based on the provided context.\",\"sources\":[]}\n"
    "Return JSON with keys: answer (string), sources (array of chunk_id).\n"
    "---\nQuestion: {query_str}\nContext:\n{context_str}"
)

qe_json = reloaded_index.as_query_engine(similarity_top_k=4, response_mode="compact",
                                         text_qa_template=prompt, similarity_cutoff=0.55)

resp = qe_json.query("How do I reset my password?")
json_out = {
    "answer": str(resp),
    "sources": [sn.node.metadata.get("chunk_id") for sn in resp.source_nodes]
}
json_out


{'answer': '{"answer":"I don\'t know based on the provided context.","sources[]}',
 'sources': ['faq.txt#chunk_000',
  'troubleshooting.md#chunk_000',
  'install_guide.md#chunk_000']}


## 7) Guardrails & calibration

- **Filters** to enforce scope/ACLs.  
- **Calibrate cutoff** (don’t hard-code): start ~0.55–0.65 for cosine.  
- **Low temperature** applies to the **LLM generation step** (not retrieval).


In [10]:

sample = qe_quick.query("Where are logs stored?")
[(round(sn.score or 0.0, 3), sn.node.metadata.get("chunk_id")) for sn in sample.source_nodes]


[(0.711, 'faq.txt#chunk_000'),
 (0.604, 'troubleshooting.md#chunk_000'),
 (0.567, 'install_guide.md#chunk_000')]


## 8) Quick evaluation: golden set (hit@k, correctness, latency)


In [11]:

from statistics import mean

gold = [
  {"q":"How to reset my password?",
   "gold_chunks":{"faq.txt#chunk_000","troubleshooting.md#chunk_000"},
   "contains":["Settings","Reset"]},
  {"q":"What are the hardware requirements?",
   "gold_chunks":{"install_guide.md#chunk_000"},
   "contains":["16GB","20GB"]},
]

def run_eval(qe, k=4):
    rows = []
    for ex in gold:
        t0=time.time(); resp = qe.query(ex["q"]); dt=time.time()-t0
        chunks = [sn.node.metadata.get("chunk_id") for sn in resp.source_nodes]
        scores = [sn.score for sn in resp.source_nodes]
        hit = len(set(chunks[:k]) & ex["gold_chunks"])>0
        ok = all(tok.lower() in str(resp).lower() for tok in ex["contains"])
        rows.append({"q":ex["q"],"hit@k":hit,"answer_ok":ok,"latency_s":dt,
                     "chunks":chunks[:k],"scores":[round(s or 0.0,3) for s in scores[:k]]})
    print("hit@k:", mean(r["hit@k"] for r in rows),
          "| acc:", mean(r["answer_ok"] for r in rows),
          "| p95 latency:", sorted(r["latency_s"] for r in rows)[int(0.95*len(rows))-1] if rows else None)
    return rows

eval_rows = run_eval(qe_json, k=4)
eval_rows


hit@k: 1 | acc: 0 | p95 latency: 0.44104909896850586


[{'q': 'How to reset my password?',
  'hit@k': True,
  'answer_ok': False,
  'latency_s': 0.6387379169464111,
  'chunks': ['faq.txt#chunk_000',
   'troubleshooting.md#chunk_000',
   'install_guide.md#chunk_000'],
  'scores': [0.7, 0.562, 0.519]},
 {'q': 'What are the hardware requirements?',
  'hit@k': True,
  'answer_ok': False,
  'latency_s': 0.44104909896850586,
  'chunks': ['install_guide.md#chunk_000',
   'troubleshooting.md#chunk_000',
   'faq.txt#chunk_000'],
  'scores': [0.696, 0.623, 0.581]}]


## 9) Serving snippet (FastAPI)

> Save the following as `fastapi_app.py` and run with `uvicorn fastapi_app:app --reload`.


In [12]:

FASTAPI_EXAMPLE = r'''
import os, time
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel
from collections import defaultdict, deque
from llama_index.core import StorageContext, load_index_from_storage

API_KEY = os.getenv("RAG_API_KEY", "dev-key")
LIMIT, WINDOW = 60, 60
buckets = defaultdict(lambda: deque())

app = FastAPI()
storage = StorageContext.from_defaults(persist_dir="index_demo")
index = load_index_from_storage(storage)
qe = index.as_query_engine(similarity_top_k=4, response_mode="compact")

@app.on_event("startup")
def warmup():
    try: _ = qe.query("warmup"); print("QE warmup OK")
    except Exception as e: print("warmup failed:", e)

class AskReq(BaseModel): question: str
class AskRes(BaseModel): answer: str; sources: list[str]

def auth_and_rate_limit(x_api_key: str | None):
    if x_api_key != API_KEY: raise HTTPException(401, "invalid api key")
    now = time.time(); q = buckets[x_api_key]
    while q and now - q[0] > WINDOW: q.popleft()
    if len(q) >= LIMIT: raise HTTPException(429, "rate limit exceeded")
    q.append(now)

@app.post("/ask", response_model=AskRes)
def ask(req: AskReq, x_api_key: str | None = Header(default=None, alias="X-API-Key")):
    auth_and_rate_limit(x_api_key)
    resp = qe.query(req.question)
    src = [sn.node.metadata.get("chunk_id") for sn in resp.source_nodes]
    return AskRes(answer=str(resp), sources=src)

@app.get("/healthz")
def healthz(): return {"ok": True}

@app.get("/readyz")
def readyz(): return {"ready": qe is not None}
'''
print(FASTAPI_EXAMPLE)



import os, time
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel
from collections import defaultdict, deque
from llama_index.core import StorageContext, load_index_from_storage

API_KEY = os.getenv("RAG_API_KEY", "dev-key")
LIMIT, WINDOW = 60, 60
buckets = defaultdict(lambda: deque())

app = FastAPI()
storage = StorageContext.from_defaults(persist_dir="index_demo")
index = load_index_from_storage(storage)
qe = index.as_query_engine(similarity_top_k=4, response_mode="compact")

@app.on_event("startup")
def warmup():
    try: _ = qe.query("warmup"); print("QE warmup OK")
    except Exception as e: print("warmup failed:", e)

class AskReq(BaseModel): question: str
class AskRes(BaseModel): answer: str; sources: list[str]

def auth_and_rate_limit(x_api_key: str | None):
    if x_api_key != API_KEY: raise HTTPException(401, "invalid api key")
    now = time.time(); q = buckets[x_api_key]
    while q and now - q[0] > WINDOW: q.popleft()
    if len(q) >= LIMIT


## 10) Troubleshooting (works for LlamaIndex / LangChain)

- Hallucinations → reduce `top_k`, lower LLM temperature, add calibrated cutoff.  
- Irrelevant → normalize & re-embed, adjust chunking.  
- Missing facts → slightly raise `k` or add reranker; confirm the fact is in the corpus.  
- Wrong citations → stabilize `chunk_id`; inspect retrieved items & scores.


In [13]:

def inspect(qe, question: str, k:int=4):
    r = qe.query(question)
    print("Answer:", str(r)[:200], "...")
    print("Top-k:", [(round(sn.score or 0.0,3), sn.node.metadata.get("chunk_id")) for sn in r.source_nodes[:k]])

inspect(qe_json, "Where are logs stored?")


Answer: {"answer":"I don't know based on the provided context.","sources[]} ...
Top-k: [(0.711, 'faq.txt#chunk_000'), (0.604, 'troubleshooting.md#chunk_000'), (0.567, 'install_guide.md#chunk_000')]



## 11) Where this differs from the LangChain lab

- LlamaIndex emphasizes **index + query engine** abstractions (persist/reload are first-class).  
- Customization uses **retrievers + postprocessors** (e.g., `SimilarityPostprocessor`), and response **modes** like `"compact"` vs `"refine"`.  
- LangChain is broader on **tool use & orchestration**; you can interop (e.g., call tools first, then fallback to a LlamaIndex QE).
